### Load Data

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler

df = pd.read_csv('/content/Day12_Used_Car_Preprocessing_Dataset.csv')
display(df.head())
display(df.info())
display(df.describe())

,Car_ID,Brand,Year,Mileage_Km,Engine_CC,Power_BHP,Fuel_Type,Transmission,City,Seller_Type,Condition,Previous_Owners,Accidents_Reported,Service_Score,Resale_Price_Lakh
0,CAR0001,Skoda,2021,69708,1152,128.8,Diesel,Manual,Lucknow,Individual,Good,1,0,72,6.38
1,CAR0002,Toyota,2020,88881,903,146.5,Diesel,Automatic,Chandigarh,Individual,Good,1,0,87,4.83
2,CAR0003,Volkswagen,2021,43646,1446,185.9,Diesel,Automatic,Hyderabad,Individual,Very Good,2,0,90,7.30
3,CAR0004,Tata,2019,70847,2069,148.8,Petrol,Manual,Lucknow,Individual,Excellent,3,0,66,3.82
4,CAR0005,Tata,2016,101228,1657,206.0,Petrol,Automatic,Ahmedabad,Dealer,Very Good,2,0,84,1.93


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 320 entries, 0 to 319
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Car_ID              320 non-null    object 
 1   Brand               320 non-null    object 
 2   Year                320 non-null    int64  
 3   Mileage_Km          320 non-null    int64  
 4   Engine_CC           320 non-null    int64  
 5   Power_BHP           320 non-null    float64
 6   Fuel_Type           320 non-null    object 
 7   Transmission        320 non-null    object 
 8   City                320 non-null    object 
 9   Seller_Type         320 non-null    object 
 10  Condition           320 non-null    object 
 11  Previous_Owners     320 non-null    int64  
 12  Accidents_Reported  320 non-null    int64  
 13  Service_Score       320 non-null    int64  
 14  Resale_Price_Lakh   320 non-null    float64
dtypes: float64(2), int64(6), object(7)
memory usage: 37.6+ KB

None

,Year,Mileage_Km,Engine_CC,Power_BHP,Previous_Owners,Accidents_Reported,Service_Score,Resale_Price_Lakh
count,320.000000,320.000000,320.000000,320.000000,320.000000,320.000000,320.000000,320.000000
mean,2019.537500,74110.203125,1346.703125,150.489688,1.668750,0.243750,76.203125,4.963031
std,3.341367,38885.260771,543.408160,36.665353,0.865369,0.528164,12.745864,3.359259
min,2014.000000,700.000000,600.000000,51.400000,1.000000,0.000000,55.000000,1.200000
25%,2017.000000,46323.250000,1004.750000,128.450000,1.000000,0.000000,64.750000,2.277500
50%,2020.000000,72718.500000,1303.000000,150.750000,1.000000,0.000000,77.000000,4.610000
75%,2022.000000,97951.500000,1635.250000,171.475000,2.000000,0.000000,87.000000,6.835000
max,2025.000000,320000.000000,5000.000000,390.000000,4.000000,2.000000,98.000000,28.500000


### Outliers

In [8]:
numeric_cols = df.select_dtypes(include=np.number).columns
Q1 = df[numeric_cols].quantile(0.25)
Q3 = df[numeric_cols].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

for col in numeric_cols:
    df[col] = df[col].clip(lower_bound[col], upper_bound[col])

### Data Split

In [9]:
X = df.drop(columns=['Car_ID', 'Resale_Price_Lakh'])
y = df['Resale_Price_Lakh']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

### Encoding

In [10]:
cond_order = [['Poor', 'Fair', 'Good', 'Very Good', 'Excellent']]
ord_enc = OrdinalEncoder(categories=cond_order)
X_train_cond = ord_enc.fit_transform(X_train[['Condition']])
X_test_cond = ord_enc.transform(X_test[['Condition']])

nominal_cols = ['Brand', 'Fuel_Type', 'Transmission', 'City', 'Seller_Type']
oh_enc = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
X_train_nom = oh_enc.fit_transform(X_train[nominal_cols])
X_test_nom = oh_enc.transform(X_test[nominal_cols])

### Scaling

In [11]:
num_features = X.select_dtypes(include=np.number).columns
scaler = StandardScaler()
X_train_num = scaler.fit_transform(X_train[num_features])
X_test_num = scaler.transform(X_test[num_features])

### Final Save

In [12]:
final_cols = list(num_features) + ['Condition'] + list(oh_enc.get_feature_names_out(nominal_cols))

train_processed = pd.DataFrame(np.hstack([X_train_num, X_train_cond, X_train_nom]), columns=final_cols, index=X_train.index)
test_processed = pd.DataFrame(np.hstack([X_test_num, X_test_cond, X_test_nom]), columns=final_cols, index=X_test.index)

train_processed['Resale_Price_Lakh'] = y_train.values
test_processed['Resale_Price_Lakh'] = y_test.values

full_processed = pd.concat([train_processed, test_processed])
full_processed.to_csv('Day12_Preprocessed_Used_Car_Dataset.csv', index=False)
display(full_processed.head())
print(full_processed.shape)

,Year,Mileage_Km,Engine_CC,Power_BHP,Previous_Owners,Accidents_Reported,Service_Score,Condition,Brand_Honda,Brand_Hyundai,...,City_Hyderabad,City_Jaipur,City_Kochi,City_Lucknow,City_Mumbai,City_Pune,Seller_Type_Certified Dealer,Seller_Type_Dealer,Seller_Type_Individual,Resale_Price_Lakh
132,-0.486391,-0.225904,-0.322825,0.303144,-0.755752,0.0,-0.454105,3.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,4.26
317,0.725445,0.089296,-0.656909,-0.328921,-0.755752,0.0,-0.614672,2.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,5.30
234,-1.395268,1.115547,-0.120148,0.418918,-0.755752,0.0,-0.534389,2.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.23
312,1.028404,-0.195102,0.481202,0.412660,0.484456,0.0,0.188165,2.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,7.09
232,-1.092309,0.706499,0.788558,-0.435309,0.484456,0.0,0.107881,3.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.69


(320, 38)


In [13]:
import pandas as pd
import numpy as np

In [14]:
df = pd.read_csv('/content/Day12_Used_Car_Preprocessing_Dataset.csv')
display(df.head())

,Car_ID,Brand,Year,Mileage_Km,Engine_CC,Power_BHP,Fuel_Type,Transmission,City,Seller_Type,Condition,Previous_Owners,Accidents_Reported,Service_Score,Resale_Price_Lakh
0,CAR0001,Skoda,2021,69708,1152,128.8,Diesel,Manual,Lucknow,Individual,Good,1,0,72,6.38
1,CAR0002,Toyota,2020,88881,903,146.5,Diesel,Automatic,Chandigarh,Individual,Good,1,0,87,4.83
2,CAR0003,Volkswagen,2021,43646,1446,185.9,Diesel,Automatic,Hyderabad,Individual,Very Good,2,0,90,7.30
3,CAR0004,Tata,2019,70847,2069,148.8,Petrol,Manual,Lucknow,Individual,Excellent,3,0,66,3.82
4,CAR0005,Tata,2016,101228,1657,206.0,Petrol,Automatic,Ahmedabad,Dealer,Very Good,2,0,84,1.93


In [15]:
df.shape

(320, 15)

In [16]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 320 entries, 0 to 319
Data columns (total 15 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Car_ID              320 non-null    object 
 1   Brand               320 non-null    object 
 2   Year                320 non-null    int64  
 3   Mileage_Km          320 non-null    int64  
 4   Engine_CC           320 non-null    int64  
 5   Power_BHP           320 non-null    float64
 6   Fuel_Type           320 non-null    object 
 7   Transmission        320 non-null    object 
 8   City                320 non-null    object 
 9   Seller_Type         320 non-null    object 
 10  Condition           320 non-null    object 
 11  Previous_Owners     320 non-null    int64  
 12  Accidents_Reported  320 non-null    int64  
 13  Service_Score       320 non-null    int64  
 14  Resale_Price_Lakh   320 non-null    float64
dtypes: float64(2), int64(6), object(7)
memory usage: 37.6+ KB

In [17]:
numeric_columns = df.select_dtypes(include=np.number).columns
Q1 = df[numeric_columns].quantile(0.25)
Q3 = df[numeric_columns].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR

for col in numeric_columns:
    df[col] = df[col].clip(lower[col], upper[col])

In [18]:
X = df.drop(columns=['Car_ID', 'Resale_Price_Lakh'])
y = df['Resale_Price_Lakh']

In [19]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [20]:
categorical_columns = X.select_dtypes(include='object').columns
numerical_features = X.select_dtypes(include=np.number).columns

In [21]:
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler

cond_order = [['Poor', 'Fair', 'Good', 'Very Good', 'Excellent']]
ord_enc = OrdinalEncoder(categories=cond_order)

X_train_cond = ord_enc.fit_transform(X_train[['Condition']])
X_test_cond = ord_enc.transform(X_test[['Condition']])

In [22]:
nominal_cols = ['Brand', 'Fuel_Type', 'Transmission', 'City', 'Seller_Type']
oh_enc = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

X_train_nom = oh_enc.fit_transform(X_train[nominal_cols])
X_test_nom = oh_enc.transform(X_test[nominal_cols])

In [23]:
scaler = StandardScaler()
X_train_num = scaler.fit_transform(X_train[numerical_features])
X_test_num = scaler.transform(X_test[numerical_features])

In [24]:
X_train_final = np.hstack([X_train_num, X_train_cond, X_train_nom])
X_test_final = np.hstack([X_test_num, X_test_cond, X_test_nom])

In [25]:
cols = list(numerical_features) + ['Condition'] + list(oh_enc.get_feature_names_out(nominal_cols))

train_df = pd.DataFrame(X_train_final, columns=cols, index=X_train.index)
test_df = pd.DataFrame(X_test_final, columns=cols, index=X_test.index)

train_df['Resale_Price_Lakh'] = y_train.values
test_df['Resale_Price_Lakh'] = y_test.values

In [26]:
processed_data = pd.concat([train_df, test_df])
processed_data.to_csv('Day12_Preprocessed_Used_Car_Dataset.csv', index=False)
display(processed_data.head())

,Year,Mileage_Km,Engine_CC,Power_BHP,Previous_Owners,Accidents_Reported,Service_Score,Condition,Brand_Honda,Brand_Hyundai,...,City_Hyderabad,City_Jaipur,City_Kochi,City_Lucknow,City_Mumbai,City_Pune,Seller_Type_Certified Dealer,Seller_Type_Dealer,Seller_Type_Individual,Resale_Price_Lakh
132,-0.486391,-0.225904,-0.322825,0.303144,-0.755752,0.0,-0.454105,3.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,4.26
317,0.725445,0.089296,-0.656909,-0.328921,-0.755752,0.0,-0.614672,2.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,5.30
234,-1.395268,1.115547,-0.120148,0.418918,-0.755752,0.0,-0.534389,2.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,1.23
312,1.028404,-0.195102,0.481202,0.412660,0.484456,0.0,0.188165,2.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,7.09
232,-1.092309,0.706499,0.788558,-0.435309,0.484456,0.0,0.107881,3.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,2.69


In [27]:
print(f'Final Shape: {processed_data.shape}')
print(f'Missing: {processed_data.isnull().sum().sum()}')

Final Shape: (320, 38)
Missing: 0
